# 08 - Hybrid classical-quantum model: gradients through an upstream classical layer

A classical `torch.nn.Linear` produces the embedding angles of a compiled PADO-Pauli program, and the
two are trained together in one backward pass. This is the property that a compiled program is an
ordinary PyTorch tensor program: `loss.backward()` reaches the classical weights, not only the
circuit angles.

```
x (2 features) -> nn.Linear(2, 4) -> embedding angles -> compiled program (thetas) -> <Z0 Z1> -> loss
```

The notebook checks three things:
1. the gradient that arrives at the classical weights is correct (against central differences),
2. joint training moves both parameter groups and the loss descends,
3. `embedding.detach()` freezes the classical layer, as documented.


- **EN** The circuit uploads the four classical outputs as `RY` rotations on two qubits, once before
  and once after a `CNOT`, then applies two trainable `RY` rotations and measures $Z_0Z_1$.
- **KO** 회로는 고전 레이어의 출력 4개를 두 큐빗의 `RY` 회전으로 올립니다. `CNOT` 앞뒤로 한 번씩 올린 뒤
  학습 가능한 `RY` 회전 두 개를 적용하고 $Z_0Z_1$을 측정합니다.


In [1]:
# EN: Setup. GPU-only, float64, fixed seed so the printed numbers are reproducible.
# KO: 초기 설정입니다. GPU 전용, float64, 시드 고정으로 출력 수치가 재현됩니다.

import warnings
warnings.filterwarnings('ignore')

import torch
from padopauli import Circuit

if not torch.cuda.is_available():
    raise RuntimeError('This notebook is configured for GPU-only execution, but CUDA is not available.')

torch.manual_seed(0)
device = 'cuda'
dtype = torch.float64
print(torch.cuda.get_device_name(0))


AMD Instinct MI300X VF


In [2]:
# EN: Compile once. embedding_idx=k reads the k-th classical output; param_idx=i reads thetas[i].
# KO: 컴파일은 한 번만 합니다. embedding_idx=k 는 고전 출력의 k번째 값을, param_idx=i 는 thetas[i] 를 읽습니다.

qc = Circuit(n_qubits=2)
qc.ry(0, embedding_idx=0)
qc.ry(1, embedding_idx=1)
qc.cnot(0, 1)
qc.ry(0, embedding_idx=2)   # data re-uploading: the same features enter again
qc.ry(1, embedding_idx=3)
qc.ry(0, param_idx=0)
qc.ry(1, param_idx=1)
qc.compile(observables=[('ZZ', [0, 1])], preset='gpu')


propagate:   0%|          | 0/7 [00:00<?, ?it/s]

[PPS Info] Propagation complete. Terms generated: 9


zero-filter:   0%|          | 0/7 [00:00<?, ?it/s]

[PPS Info] Terms retained after pruning: 3 (33.333333% of peak)


CompiledProgram(circuit=[RY[0](prior[0]), RY[1](prior[1]), CNOT[0, 1], RY[0](prior[2]), RY[1](prior[3]), RY[0](θ[0]), RY[1](θ[1])], psum_union=TensorOperatorProgram(n_qubits=2, x_mask=tensor([0, 0, 0], device='cuda:0'), z_mask=tensor([2, 3, 1], device='cuda:0'), coeff_init=tensor([1.], device='cuda:0', dtype=torch.float64), steps=[TensorSparseStep(mat_const=tensor(indices=tensor([], size=(2, 0)),
       values=tensor([], size=(0,)),
       device='cuda:0', size=(2, 1), nnz=0, dtype=torch.float64,
       layout=torch.sparse_coo), mat_cos=tensor(indices=tensor([], size=(2, 0)),
       values=tensor([], size=(0,)),
       device='cuda:0', size=(2, 1), nnz=0, dtype=torch.float64,
       layout=torch.sparse_coo), mat_sin=tensor(indices=tensor([[1],
                       [0]]),
       values=tensor([-1.]),
       device='cuda:0', size=(2, 1), nnz=1, dtype=torch.float64,
       layout=torch.sparse_coo), param_idx=1, shape=(2, 1), embedding_idx=-1, unchanged_cols=tensor([0], device='cuda:0'),

In [3]:
# EN: The hybrid model. lin is the upstream classical layer; thetas are the circuit angles.
# KO: 하이브리드 모델입니다. lin 이 상류 고전 레이어이고 thetas 는 회로 각도입니다.

lin = torch.nn.Linear(2, 4).to(device=device, dtype=dtype)
thetas = torch.nn.Parameter(torch.zeros(2, dtype=dtype, device=device))

X = (torch.rand(256, 2, dtype=dtype, device=device) * 2 - 1)
y = ((X[:, 0] * X[:, 1]) > 0).to(dtype)          # XOR-like target

def loss_fn():
    angles = lin(X)                               # classical layer -> embedding angles
    expv = qc.expvals(thetas, embedding=angles)[:, 0]
    prob = torch.clamp((expv + 1) * 0.5, 1e-6, 1 - 1e-6)
    return torch.nn.functional.binary_cross_entropy(prob, y)

print('loss at initialization:', float(loss_fn()))


loss at initialization: 1.7090882410369925


## 1) The gradient reaching the classical weights is correct

One backward pass, then every entry of `lin.weight` is compared against a central difference of the
same loss. Agreement at this level means the chain rule is closed through the compiled program.


In [4]:
# EN: Analytic gradient from one backward pass, then central differences on the same weights.
# KO: backward 한 번으로 얻은 해석적 gradient 를 같은 가중치에 대한 중앙차분과 비교합니다.

for t in (lin.weight, lin.bias, thetas):
    t.grad = None
loss_fn().backward()
g_analytic = lin.weight.grad.detach().clone()
g_theta = thetas.grad.detach().clone()

h = 1e-6
g_fd = torch.zeros_like(g_analytic)
with torch.no_grad():
    for i in range(g_analytic.shape[0]):
        for j in range(g_analytic.shape[1]):
            original = lin.weight[i, j].item()
            lin.weight[i, j] = original + h; loss_plus = float(loss_fn())
            lin.weight[i, j] = original - h; loss_minus = float(loss_fn())
            lin.weight[i, j] = original
            g_fd[i, j] = (loss_plus - loss_minus) / (2 * h)

print(f'max |analytic - central difference| over the classical weights: {(g_analytic - g_fd).abs().max():.2e}')
print(f'smallest |analytic| among the 8 weights:                        {g_analytic.abs().min():.4f}')
print(f'gradient norm on the circuit angles:                            {g_theta.norm():.4f}')


max |analytic - central difference| over the classical weights: 1.22e-10
smallest |analytic| among the 8 weights:                        0.0167
gradient norm on the circuit angles:                            1.9119


## 2) Joint training

One optimizer over both parameter groups. Both move, and the loss descends.


In [5]:
# EN: Train the classical layer and the circuit angles together.
# KO: 고전 레이어와 회로 각도를 함께 학습합니다.

W0 = lin.weight.detach().clone()
theta0 = thetas.detach().clone()
opt = torch.optim.Adam([{'params': lin.parameters()}, {'params': [thetas]}], lr=0.05)

loss_before = float(loss_fn())
for step in range(300):
    opt.zero_grad(set_to_none=True)
    loss = loss_fn()
    loss.backward()
    opt.step()
loss_after = float(loss_fn())

with torch.no_grad():
    prob = torch.clamp((qc.expvals(thetas, embedding=lin(X))[:, 0] + 1) * 0.5, 1e-6, 1 - 1e-6)
    accuracy = ((prob > 0.5).to(dtype) == y).to(dtype).mean().item()

print(f'loss {loss_before:.4f} -> {loss_after:.4f}, training-set accuracy {accuracy * 100:.1f}%')
print(f'|change in classical weights| = {(lin.weight.detach() - W0).norm():.3f}')
print(f'|change in circuit angles|    = {(thetas.detach() - theta0).norm():.3f}')


loss 1.7091 -> 0.2450, training-set accuracy 97.3%
|change in classical weights| = 4.387
|change in circuit angles|    = 0.939


## 3) Freezing the upstream layer

Passing `embedding.detach()` cuts the classical layer out of the graph, so it receives no gradient
while the circuit angles still train.


In [6]:
# EN: The documented way to freeze an upstream layer.
# KO: 상류 레이어를 고정하는 문서화된 방법입니다.

lin.zero_grad(set_to_none=True)
prob = torch.clamp((qc.expvals(thetas, embedding=lin(X).detach())[:, 0] + 1) * 0.5, 1e-6, 1 - 1e-6)
torch.nn.functional.binary_cross_entropy(prob, y).backward()
print('classical weight gradient with embedding.detach():', lin.weight.grad)


classical weight gradient with embedding.detach(): None
